# 26.9.17

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
import numpy as np
from datasets import load_dataset

news = load_dataset('argilla/news-summary',split = 'test')
df = news.to_pandas().sample(5000, random_state = 42)[['text', 'prediction']]
df['text'] = 'summarize: ' + df['text']
df['prediction'] = df['prediction'].map(lambda x: x[0]['text'])
train, valid, test = np.split(df.sample(frac = 1, random_state = 42), [int(0.6 * len(df)), int(0.8 * len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Putin says had useful interaction with Trump at Vi
3000
1000
1000


c:\Users\wm032\Documents\computervision\.venv\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [4]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    source = tokenizer(text = data.text.tolist(), padding = 'max_length', max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    target = tokenizer(text = data.prediction.tolist(), padding = 'max_length', max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    source_ids = source['input_ids'].squeeze().to(device)
    source_mask = source['attention_mask'].squeeze().to(device)
    target_ids = target['input_ids'].squeeze().to(device)
    target_mask = target['attention_mask'].squeeze().to(device)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset,
                            sampler = data_sampler,
                            batch_size = batch_size)
    return dataloader

In [5]:
epochs = 5
batch_size = 8

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path = 't5-small')


train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\wm032\Documents\computervision\.venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wm032\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[tensor([[21603,    10,  6045,  ...,    37, 16870,     1],
        [21603,    10,   309,  ...,   718,    21,     1],
        [21603,    10,   454,  ..., 26988,   789,     1],
        ...,
        [21603,    10,  7109,  ...,     0,     0,     0],
        [21603,    10,     3,  ...,  6653,  3272,     1],
        [21603,    10,  3001,  ...,  1103,    13,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[16870, 12346,  1088,  ...,     0,     0,     0],
        [28819,   845,   350,  ...,     0,     0,     0],
        [13052,  2839,     6,  ...,     0,     0,     0],
        ...,
        [15363,  2053,  1338,  ...,     0,     0,     0],
        [ 5199,   159,   223,  ...,     0,     0,     0],
        [ 3411,    12,   918,  ...,     0,     0,     0]], device='cuda:0'), ten

In [6]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path = 't5-small').to(device)

optimizer = optim.AdamW(model.parameters(), lr = 1e-5, eps = 1e-8)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [8]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis= 1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train_model(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask = source_mask, decoder_input_ids = decoder_input_ids, labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(input_ids = source_ids, attention_mask = source_mask, decoder_input_ids = decoder_input_ids, labels = labels)
            
            loss = outputs.loss
            val_loss += loss.item()
        
        val_loss = val_loss / len(dataloader)
        return val_loss


In [9]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train_model(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)

    print(epoch + 1, train_loss, val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "./models/T5ForConditionalGeneration.pt")
        print("Saved!!")

1 4.320184160868327 3.33073653793335
Saved!!
2 3.422196943283081 2.916817729949951
Saved!!
3 3.129735933303833 2.767329840660095
Saved!!
4 3.00757443968455 2.6741956758499144
Saved!!
5 2.8910997482935588 2.611466507911682
Saved!!
